<a id="inicio"></a>

# Heart Disease Classifier — MLOps Pipeline

## Objective

Build a full MLOps pipeline for a heart-disease risk classifier — from raw data through model training with experiment tracking, into a production-ready Streamlit application. The emphasis is on **operational correctness**: every experiment is tracked, every model is versioned, and deployment follows reproducible patterns.

## Methodology

- **Data**: [Heart Disease dataset from Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) — 918 patient records with 7 predictive features
- **Experiment tracking**: [MLflow](https://mlflow.org/) for parameter, metric, and model artifact logging
- **Model training**: Two parallel approaches — `RandomForestClassifier` (scikit-learn) and a dense neural network (TensorFlow/Keras) — to compare tree-based and NN approaches under the same tracking infrastructure
- **Model registry**: Promote the best model to production via the `prod` alias in MLflow's registry
- **Deployment**: Ship the registered model into a [Streamlit](https://streamlit.io/) app using two patterns — embedded (model loaded inside the Streamlit process) or API-backed (FastAPI service consumed by Streamlit)

## Clinical context

Cardiovascular disease is the leading cause of death globally — roughly 17.9 million deaths per year, or 31% of all deaths worldwide. People at elevated cardiovascular risk (hypertension, diabetes, hyperlipidemia) benefit from early detection. A binary classifier that flags high-risk patients for further workup can support this preventive screening.

## Repository layout

- `Capstone XIV.ipynb` — the training + registration notebook (this file)
- `api.py` — FastAPI service that serves the MLflow `prod` model
- `app_api.py` — Streamlit front-end that consumes the FastAPI
- `embedido.py` — Streamlit alternative that embeds the model directly
- `data/heart.csv` — training data
- `images/` — architecture and app screenshots

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. Clone the repo and create a Python 3.9+ virtual environment:
   `python -m venv .venv && source .venv/bin/activate`
2. Install dependencies (notebook + deployment apps):
   `pip install mlflow scikit-learn tensorflow pandas matplotlib streamlit fastapi uvicorn`
3. **Start MLflow**: in a separate terminal, run `mlflow server --host 0.0.0.0 --port 5000` — keep it running throughout
4. Launch Jupyter and execute this notebook top-to-bottom. It will:
   - Load and split the dataset
   - Train a Random Forest and a Dense NN, logging each run to MLflow
   - Promote the chosen model to the `prod` alias in the MLflow Registry
5. **Launch the deployment apps**:
   - For the embedded pattern: `streamlit run embedido.py`
   - For the API pattern: `uvicorn api:app --reload` (terminal 1) + `streamlit run app_api.py` (terminal 2)

All apps read from `http://localhost:5000` (MLflow) — adjust as needed for remote deployments.

</div>

---

---

## Contents

1. **Data** — loading and cleaning
2. **MLOps** — training models with MLflow experiment tracking and the model registry
3. **Deployment** — shipping the registered model into a Streamlit application

Every section includes all guidance needed to complete it.

In [1]:
import pandas as pd
import mlflow
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

import tensorflow as tf

## 1. Data

The dataset covers cardiovascular disease screening — 918 patients with 12 recorded features. We use 7 features to predict the binary target `HeartDisease`:

| Feature | Description |
|---|---|
| `Age` | Patient age in years |
| `Sex` | Biological sex (`M` / `F`) |
| `RestingBP` | Resting blood pressure (mmHg) |
| `Cholesterol` | Serum cholesterol (mg/dl) |
| `FastingBS` | Fasting blood sugar flag — `1` if > 120 mg/dl, `0` otherwise |
| `MaxHR` | Maximum heart rate achieved during stress test |
| `HeartDisease` | Target — `1` if heart disease, `0` if normal |

### 1.1 Load the data and keep only the feature columns

In [2]:
data_path = "data/heart.csv"
feature_columns = ["Age", "Sex", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "HeartDisease"]
df = pd.read_csv(data_path, usecols=feature_columns)
df.head()

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,HeartDisease
0,40,M,140,289,0,172,0
1,49,F,160,180,0,156,1
2,37,M,130,283,0,98,0
3,48,F,138,214,0,108,1
4,54,M,150,195,0,122,0


### 1.2 Encode `Sex` as numeric — `M: 0`, `F: 1`

In [3]:
df["Sex"] = df["Sex"].map({"M": 0, "F": 1})


### 1.3 Split features from target

In [4]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]


### 1.4 Train/test split (25% held out)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

## 2. MLOps

In this section we use **MLflow** to track experiments and version models — establishing a proper audit trail of what was tried and what ended up in production.

Two candidate algorithms:

- **`RandomForestClassifier`** (scikit-learn)
- **Dense neural network** (TensorFlow/Keras)

For each approach, we:

1. Create an MLflow experiment
2. Launch a tracked run that logs:
   - The hyperparameters used
   - The trained model
   - Train/test metrics
3. After the first run, optionally iterate on the hyperparameters and launch a second run to compare

Throughout, use the MLflow UI (`mlflow server`) to inspect run comparisons.

**Training-flow diagram:**

![MLflow training architecture](images/mlflow-training.png)

### 2.0 Point MLflow at the tracking server

Set the URI of the running MLflow server. Default is `http://localhost:5000`.

In [6]:
mlflow.set_tracking_uri("http://localhost:5000")

### 2.1 Model training + experiment logging

#### Option 1: Random Forest with scikit-learn

Train a `RandomForestClassifier` with the following hyperparameters (and log them to MLflow):

- `n_estimators = 124`
- `min_samples_split = 10`
- `min_samples_leaf = 2`
- `max_features = 'sqrt'`
- `max_depth = 90`
- `bootstrap = False`

Log these metrics:

- `train_accuracy`, `train_precision`, `train_recall`
- `test_accuracy`, `test_precision`, `test_recall`

In [21]:
mlflow.set_experiment("heart-disease-classifier")

params = {
    "n_estimators": 124,
    "min_samples_split": 10,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "max_depth": 90,
    "bootstrap": False
}

with mlflow.start_run():
    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_train, y_train)


    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mlflow.log_param("model_type", "RandomForest")
    for key, value in params.items():
        mlflow.log_param(key, value)

    mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
    mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
    mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))

    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))

    mlflow.sklearn.log_model(model, "model",registered_model_name="model_random_forest")




2025/06/28 16:35:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/28 16:35:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'model_random_forest'.
2025/06/28 16:35:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_random_forest, version 1
Created version '1' of model 'model_random_forest'.


🏃 View run fortunate-sheep-10 at: http://localhost:5000/#/experiments/461735160528471256/runs/3967f3aeaa3b4907a4962d1f5347d7d5
🧪 View experiment at: http://localhost:5000/#/experiments/461735160528471256


#### Option 2: Dense Neural Network with TensorFlow

Train a sequential dense NN with the following hyperparameters (and log them):

- `n_layers = 4`
- `nodes = [32, 32, 16, 8]`
- `optimizer = 'adam'`
- `loss = 'binary_crossentropy'`

Log the same six metrics as Option 1.

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

n_layers = 4
nodes = [32, 32, 16, 8]
optimizer = 'adam'
loss = 'binary_crossentropy'
epochs = 50
batch_size = 16

with mlflow.start_run():
    # Construir modelo secuencial
    model = Sequential()
    model.add(Dense(nodes[0], activation='relu', input_shape=(X_train.shape[1],)))
    for n in nodes[1:]:
        model.add(Dense(n, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  
    model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

 
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)


    y_train_pred = (model.predict(X_train) > 0.5).astype("int32")
    y_test_pred = (model.predict(X_test) > 0.5).astype("int32")

   
    mlflow.log_param("model_type", "DenseNN")
    mlflow.log_param("n_layers", n_layers)
    mlflow.log_param("nodes", nodes)
    mlflow.log_param("optimizer", optimizer)
    mlflow.log_param("loss", loss)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", batch_size)

    mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
    mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
    mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))

    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))

   
    mlflow.tensorflow.log_model(model, "model_tensorflow",registered_model_name="model_tensorflow")

c:\Users\usuario\anaconda3\envs\.caps81\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


2025/06/28 16:36:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/28 16:36:39 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2025/06/28 16:36:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'model_tensorflow'.
2025/06/28 16:36:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_tensorflow, version 1
Created version '1' of model 'model_tensorflow'.


🏃 View run languid-steed-137 at: http://localhost:5000/#/experiments/461735160528471256/runs/24bb42c1b2244a308ceacb858b4a5fdf
🧪 View experiment at: http://localhost:5000/#/experiments/461735160528471256


### 2.2 Register the production model

Register the selected model with the `prod` alias so downstream apps can resolve it by name + alias rather than by run ID.

**Tip**: Before shipping to production, sanity-check predictions from the `prod`-aliased model — a final smoke test that the registry lookup works end-to-end.

In [24]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.set_registered_model_alias("model_random_forest", "prod", "1")


In [28]:
import mlflow.pyfunc
model = mlflow.pyfunc.load_model("models:/model_random_forest@prod")
data = pd.DataFrame([{
    "Age": 50,
    "Sex": 0,              # 0: Male, 1: Female
    "RestingBP": 130,
    "Cholesterol": 250,
    "FastingBS": 1,        # 1: fasting blood sugar > 120 mg/dl
    "MaxHR": 150
}])
prediction = model.predict(data)
print(f"Prediction: {prediction[0]}")

c:\Users\usuario\anaconda3\envs\.caps81\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Predicción: 0


## 3. Deployment

The registered model ships into a Streamlit application. The app should look like:

![app screenshot](images/app.png)

Internally the app uses the production model to predict heart-disease risk from user-entered features. Two valid deployment patterns:

![deployment patterns](images/deployment.png)

- **Embedded** (`embedido.py`) — Streamlit loads the model directly in its own process. Simpler architecture, but couples the front-end to the model runtime.
- **API-backed** (`api.py` + `app_api.py`) — A FastAPI service wraps the model; Streamlit calls it over HTTP. More complex, but standard for production because the model service can be scaled independently of the UI layer.